In [ ]:
import duckdb

profile = duckdb.sql("""
    SELECT
        regexp_extract(filename, '([^/\\\\]+)\\.csv', 1) AS city,
        COUNT(*) AS n_listings,
        COUNT(DISTINCT room_type) AS n_room_types,
        ROUND((MAX(latitude) - MIN(latitude)) * 111, 0) AS span_ns_km,
        ROUND(MAX(price) - MIN(price), 0) AS price_range,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        COUNT(DISTINCT neighbourhood) AS n_neighbourhoods,
        MAX(last_review) AS latest_review
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    GROUP BY city
    ORDER BY n_listings DESC
""").df()

print(profile.shape)
profile.head(20)

In [3]:
import pandas as pd

index=pd.read_csv("../data/city_index.csv")

removed=index[index["active_entire_homes"]<500]

print(removed)

               city  total_listings  entire_homes  active_entire_homes
116       cambridge            1457           775                  463
117         bozeman             591           527                  414
118       singapore            3247          1226                  328
119          albany             490           345                  288
120  barossa-valley             360           316                  263
121        salem-or             349           251                  193
122   pacific-grove             270           224                  149


In [5]:
import duckdb

profile = duckdb.sql("""
    SELECT
        regexp_extract(filename, '([^/\\\\]+)\\.csv', 1) AS city_file,
        COUNT(*) AS n_listings,
        COUNT(DISTINCT room_type) AS n_room_types,
        COUNT(DISTINCT neighbourhood) AS n_neighbourhoods,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        SUM(CASE WHEN number_of_reviews_ltm > 0 THEN 1 ELSE 0 END) AS active,
        ROUND(MEDIAN(price), 0) AS median_price,
        MAX(last_review) AS latest_review
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    GROUP BY regexp_extract(filename, '([^/\\\\]+)\\.csv', 1)
    ORDER BY n_listings DESC
""").df()

print(profile.shape)
profile.head(20)

(125, 8)


,city_file,n_listings,n_room_types,n_neighbourhoods,null_price,active,median_price,latest_review
0,london,92799,4,33,30402.0,47174.0,180.0,2026-06-30
1,paris,77679,4,20,29277.0,41306.0,206.0,2026-06-28
2,sicily,56877,4,372,5009.0,28838.0,126.0,2026-07-02
3,new-zealand,50932,4,210,5223.0,40810.0,264.0,2026-06-22
4,rio-de-janeiro,48729,4,154,4172.0,33080.0,453.0,2026-06-30
5,puglia,48685,4,248,5296.0,23051.0,140.0,2026-07-03
6,los-angeles,43932,4,265,5570.0,24228.0,224.0,2026-06-22
7,sao-paulo,42378,4,96,526.0,33664.0,331.0,2026-06-16
8,hawaii,38231,4,30,4105.0,24022.0,387.0,2026-07-01
9,rome,37228,4,15,2761.0,27595.0,162.0,2026-07-06


In [6]:
duckdb.sql("""
    SELECT
        CASE WHEN number_of_reviews_ltm > 0 THEN 'active' ELSE 'inactive' END AS status,
        COUNT(*) AS n,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS nulls,
        ROUND(100.0 * SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_null
    FROM read_csv_auto('../data/london.csv')
    GROUP BY 1
""")

┌──────────┬───────┬────────┬──────────┐
│  status  │   n   │ nulls  │ pct_null │
│ varchar  │ int64 │ int128 │  double  │
├──────────┼───────┼────────┼──────────┤
│ active   │ 47174 │   6055 │     12.8 │
│ inactive │ 45625 │  24347 │     53.4 │
└──────────┴───────┴────────┴──────────┘

In [8]:
duckdb.sql("""
    SELECT *
    FROM read_csv_auto('../data/london.csv')
""")

┌──────────┬───────────────────────────────────────────────────┬───────────┬─────────────────────┬──────────────┬─────────────────────┬────────────────────────┬───────────────────┬──────────────────────┬─────────────────┬───────┬────────────────┬───────────────────┬─────────────┬───────────────────┬────────────────────────────────┬──────────────────┬───────────────────────┬─────────┐
│    id    │                       name                        │  host_id  │   host_profile_id   │  host_name   │ neighbourhood_group │     neighbourhood      │     latitude      │      longitude       │    room_type    │ price │ minimum_nights │ number_of_reviews │ last_review │ reviews_per_month │ calculated_host_listings_count │ availability_365 │ number_of_reviews_ltm │ license │
│  int64   │                      varchar                      │   int64   │        int64        │   varchar    │       varchar       │        varchar         │      double       │        double        │     varchar     │ int6

In [9]:
duckdb.sql("""DESCRIBE SELECT * FROM read_csv_auto('../data/london.csv')""")

┌────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │ column_type │  null   │   key   │ default │  extra  │
│            varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name                           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ host_id                        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ host_profile_id                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ host_name                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ neighbourhood_group            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ neighbourhood                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude           

In [10]:
duckdb.sql(""" SUMMARIZE SELECT * FROM read_csv_auto('../data/london.csv')""")

┌────────────────────────────────┬─────────────┬────────────────────────────────────────────┬──────────────────────────────────────┬───────────────┬───────────────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────────┬───────┬─────────────────┐
│          column_name           │ column_type │                    min                     │                 max                  │ approx_unique │            avg            │          std           │         q25          │         q50         │         q75          │ count │ null_percentage │
│            varchar             │   varchar   │                  varchar                   │               varchar                │     int64     │          varchar          │        varchar         │       varchar        │       varchar       │       varchar        │ int64 │  decimal(9,2)   │
├────────────────────────────────┼─────────────┼────────────────────────────────────────────┼───────────────────

In [ ]:
duckdb.sql("""
    SELECT 

_IncompleteInputError: incomplete input (1410792519.py, line 1)

In [5]:
import duckdb

duckdb.sql("""
    SELECT 
        parse_filename(filename, true) AS city,
        COUNT(*) AS N_listings,
        MEDIAN(price) AS median_price,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        ROUND(100.0*SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_null_price
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    GROUP BY parse_filename(filename, true)
    ORDER BY pct_null_price DESC
    """).df()

,city,N_listings,median_price,null_price,pct_null_price
0,cities_filtered,116,NaN,116.0,100.0
1,city_index,123,NaN,123.0,100.0
2,zurich,3308,0.0,1680.0,50.8
3,lyon,9355,104.0,3892.0,41.6
4,copenhagen,23146,1689.0,9284.0,40.1
...,...,...,...,...,...
120,western-australia,11724,324.0,177.0,1.5
121,nairobi,22008,6147.0,314.0,1.4
122,sao-paulo,42378,331.0,526.0,1.2
123,bogota,19187,161460.0,195.0,1.0


In [7]:
duckdb.sql("""
    SELECT 
        parse_filename(filename, true) AS city,
        COUNT(*) AS N_listings,
        MEDIAN(price) AS median_price,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        ROUND(100.0*SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_null_price
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    GROUP BY parse_filename(filename, true)
    ORDER BY pct_null_price DESC
    LIMIT 50
    """).df()

,city,N_listings,median_price,null_price,pct_null_price
0,cities_filtered,116,NaN,116.0,100.0
1,city_index,123,NaN,123.0,100.0
2,zurich,3308,0.0,1680.0,50.8
3,lyon,9355,104.0,3892.0,41.6
4,copenhagen,23146,1689.0,9284.0,40.1
5,geneva,2604,0.0,1004.0,38.6
6,amsterdam,10465,287.0,3994.0,38.2
7,paris,77679,206.0,29277.0,37.7
8,vaud,5301,0.0,1998.0,37.7
9,munich,6890,145.0,2425.0,35.2


In [8]:
duckdb.sql("""
    SELECT
    CASE WHEN price IS NULL THEN 'null price' ELSE 'has price' END AS grp,
    COUNT(*) AS n,
    ROUND(AVG(availability_365), 1) AS avg_avail,
    ROUND(AVG(number_of_reviews_ltm), 1) AS avg_reviews_ltm
FROM read_csv_auto('../data/paris.csv')
GROUP BY grp""")

┌────────────┬───────┬───────────┬─────────────────┐
│    grp     │   n   │ avg_avail │ avg_reviews_ltm │
│  varchar   │ int64 │  double   │     double      │
├────────────┼───────┼───────────┼─────────────────┤
│ has price  │ 48402 │     195.2 │            10.3 │
│ null price │ 29277 │      40.0 │             2.0 │
└────────────┴───────┴───────────┴─────────────────┘

In [9]:
duckdb.sql("""
    SELECT
    CASE WHEN price IS NULL THEN 'null price' ELSE 'has price' END AS grp,
    COUNT(*) AS n_all,
    SUM(CASE WHEN number_of_reviews_ltm > 0 THEN 1 ELSE 0 END) AS n_active
FROM read_csv_auto('../data/paris.csv')
GROUP BY grp""")

┌────────────┬───────┬──────────┐
│    grp     │ n_all │ n_active │
│  varchar   │ int64 │  int128  │
├────────────┼───────┼──────────┤
│ null price │ 29277 │     6833 │
│ has price  │ 48402 │    34473 │
└────────────┴───────┴──────────┘

In [11]:
duckdb.sql("""
SELECT
    parse_filename(filename, true) AS city,
    SUM(CASE WHEN number_of_reviews_ltm > 0 AND price IS NOT NULL THEN 1 ELSE 0 END) AS active_priced,
    SUM(CASE WHEN number_of_reviews_ltm > 0 AND price IS NULL     THEN 1 ELSE 0 END) AS active_null,
    ROUND(100.0 * SUM(CASE WHEN number_of_reviews_ltm > 0 AND price IS NULL THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN number_of_reviews_ltm > 0 THEN 1 ELSE 0 END), 0), 1) AS pct_lost
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
GROUP BY parse_filename(filename, true)
ORDER BY pct_lost DESC
LIMIT 50
""").df()

,city,active_priced,active_null,pct_lost
0,zurich,1112.0,699.0,38.6
1,copenhagen,10464.0,3964.0,27.5
2,geneva,1051.0,348.0,24.9
3,amsterdam,5081.0,1482.0,22.6
4,vaud,2186.0,561.0,20.4
5,mallorca,8140.0,1953.0,19.4
6,munich,3116.0,670.0,17.7
7,oslo,6750.0,1419.0,17.4
8,stockholm,2338.0,477.0,16.9
9,paris,34473.0,6833.0,16.5


In [13]:
duckdb.sql("""SELECT price, COUNT(*) AS n
FROM read_csv_auto('../data/zurich.csv')
GROUP BY price ORDER BY n DESC LIMIT 10""")

┌───────┬───────┐
│ price │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  NULL │  1680 │
│     0 │  1532 │
│     1 │    96 │
└───────┴───────┘

In [14]:
duckdb.sql("""SELECT price, COUNT(*) AS n
FROM read_csv_auto('../data/geneva.csv')
GROUP BY price ORDER BY n DESC LIMIT 10""")

┌───────┬───────┐
│ price │   n   │
│ int64 │ int64 │
├───────┼───────┤
│     0 │  1475 │
│  NULL │  1004 │
│     1 │   125 │
└───────┴───────┘

In [15]:
duckdb.sql("""SELECT price, COUNT(*) AS n
FROM read_csv_auto('../data/vaud.csv')
GROUP BY price ORDER BY n DESC LIMIT 10""")

┌───────┬───────┐
│ price │   n   │
│ int64 │ int64 │
├───────┼───────┤
│     0 │  3017 │
│  NULL │  1998 │
│     1 │   286 │
└───────┴───────┘

In [16]:
duckdb.sql("""SELECT
    parse_filename(filename, true) AS city,
    COUNT(*) AS n,
    SUM(CASE WHEN price <= 1 THEN 1 ELSE 0 END) AS n_junk,
    ROUND(100.0 * SUM(CASE WHEN price <= 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_junk,
    COUNT(DISTINCT price) AS n_distinct_prices
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
GROUP BY parse_filename(filename, true)
ORDER BY pct_junk DESC
""").df()

,city,n,n_junk,pct_junk,n_distinct_prices
0,vaud,5301,3303.0,62.3,2
1,geneva,2604,1600.0,61.4,2
2,zurich,3308,1628.0,49.2,2
3,fort-worth,2419,0.0,0.0,601
4,malaga,9573,0.0,0.0,720
...,...,...,...,...,...
120,london,92799,0.0,0.0,1701
121,victoria,3531,0.0,0.0,829
122,budapest,11459,0.0,0.0,7481
123,pays-basque,12096,0.0,0.0,961


In [21]:
duckdb.sql("""SELECT
    parse_filename(filename, true) AS city,
    COUNT(*) AS n,
    SUM(CASE WHEN price <= 1 THEN 1 ELSE 0 END) AS n_junk,
    ROUND(100.0 * SUM(CASE WHEN price <= 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_junk,
    COUNT(DISTINCT price) AS n_distinct_prices
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
GROUP BY parse_filename(filename, true)
ORDER BY n_distinct_prices 

""").df()

,city,n,n_junk,pct_junk,n_distinct_prices
0,city_index,123,0.0,0.0,0
1,cities_filtered,116,0.0,0.0,0
2,zurich,3308,1628.0,49.2,2
3,geneva,2604,1600.0,61.4,2
4,vaud,5301,3303.0,62.3,2
...,...,...,...,...,...
120,istanbul,26631,0.0,0.0,7961
121,bogota,19187,0.0,0.0,9444
122,santiago,18534,0.0,0.0,10131
123,tokyo,34419,0.0,0.0,14359
